# Music Annotation Project V5.2: Production Annotation Removal

**Runtime:** T4 GPU for YOLO mask generation; reconstruction itself is CPU-based.

This version follows the controlled-benchmark findings:
- **biharmonic** is the provisional final-quality default;
- **Telea** is retained as a fast preview;
- **Navier–Stokes** is retained as a symbol-edge alternative;
- staff continuation is conservative and experimental;
- every output is byte-for-byte unchanged outside the removal mask;
- no clean page or hidden ground truth is used.

A fully obscured musical symbol cannot be recovered with certainty from an annotated-only scan. The notebook therefore saves uncertainty and review outputs rather than claiming exact recovery.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics opencv-python-headless scikit-image pandas matplotlib


## Configuration


In [ ]:
from ultralytics import YOLO
import torch, pandas as pd
import matplotlib.pyplot as plt

import cv2, json, hashlib, shutil, math, os
import numpy as np
from pathlib import Path
from datetime import datetime, timezone
from skimage.restoration import inpaint as sk_inpaint

SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def image_files(folder): return sorted(p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED) if Path(folder).exists() else []
def md5(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def object_hash(value): return hashlib.md5(json.dumps(value,sort_keys=True).encode()).hexdigest()
def immutable(original,candidate,mask):
    out=original.copy(); out[mask>0]=candidate[mask>0]
    assert np.array_equal(out[mask==0],original[mask==0])
    return out

def padded_components(mask,margin=40,min_area=8):
    n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
    boxes=[];h,w=mask.shape
    for i in range(1,n):
        x,y,bw,bh,a=stats[i]
        if a<min_area: continue
        boxes.append((max(0,x-margin),max(0,y-margin),min(w,x+bw+margin),min(h,y+bh+margin)))
    # merge overlapping padded rectangles
    changed=True
    while changed:
        changed=False; merged=[]
        while boxes:
            a=boxes.pop(); ax0,ay0,ax1,ay1=a; rest=[]
            for b in boxes:
                bx0,by0,bx1,by1=b
                if not (ax1<bx0 or bx1<ax0 or ay1<by0 or by1<ay0):
                    ax0=min(ax0,bx0);ay0=min(ay0,by0);ax1=max(ax1,bx1);ay1=max(ay1,by1);changed=True
                else: rest.append(b)
            boxes=rest;merged.append((ax0,ay0,ax1,ay1))
        boxes=merged
    return boxes

def crop_inpaint(image,mask,method,margin=40):
    result=image.copy()
    for x0,y0,x1,y1 in padded_components(mask,margin):
        im=image[y0:y1,x0:x1]; mk=mask[y0:y1,x0:x1]
        if method=='telea': cand=cv2.inpaint(im,mk,3,cv2.INPAINT_TELEA)
        elif method=='navier_stokes': cand=cv2.inpaint(im,mk,5,cv2.INPAINT_NS)
        elif method=='biharmonic':
            z=sk_inpaint.inpaint_biharmonic(im.astype(np.float32)/255.0,mk>0,channel_axis=None)
            cand=np.clip(z*255,0,255).astype(np.uint8)
        else: raise ValueError(method)
        local=result[y0:y1,x0:x1];local[mk>0]=cand[mk>0];result[y0:y1,x0:x1]=local
    return immutable(image,result,mask)

def staff_evidence(observed,mask):
    visible=observed.copy();visible[mask>0]=255
    ink=(visible<190).astype(np.uint8)*255
    long=cv2.morphologyEx(ink,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))
    return cv2.dilate(long,np.ones((3,7),np.uint8))>0

def safe_staff_continuation(observed,mask,base,max_gap=80):
    out=base.copy(); evidence=staff_evidence(observed,mask); h,w=observed.shape
    rows=np.where(evidence.sum(axis=1)>=18)[0]
    for y in rows:
        xs=np.where(mask[y]>0)[0]
        if not xs.size: continue
        for start in np.split(xs,np.where(np.diff(xs)>1)[0]+1):
            if not len(start) or len(start)>max_gap: continue
            x0,x1=int(start[0]),int(start[-1]); r=16
            left=evidence[max(0,y-1):min(h,y+2),max(0,x0-r):x0]
            right=evidence[max(0,y-1):min(h,y+2),x1+1:min(w,x1+r+1)]
            if left.sum()<2 or right.sum()<2: continue
            vals=[]
            for yy in range(max(0,y-1),min(h,y+2)):
                for xa,xb in [(max(0,x0-r),x0),(x1+1,min(w,x1+r+1))]:
                    q=observed[yy,xa:xb]; vals.extend(q[q<190].tolist())
            if not vals: continue
            value=min(int(np.percentile(vals,35)),140)
            segment=(mask[max(0,y-1):min(h,y+2),x0:x1+1]>0)
            patch=out[max(0,y-1):min(h,y+2),x0:x1+1]
            patch[segment]=value
    return immutable(observed,out,mask)

CONFIG_PATH=Path('/content/drive/MyDrive/MusicAnnotationProject/project_config.json')
CONFIG=json.loads(CONFIG_PATH.read_text()) if CONFIG_PATH.exists() else {'project_root':'/content/drive/MyDrive/MusicAnnotationProject'}
ROOT=Path(CONFIG['project_root'])
SOURCE_DIR=ROOT/'aligned'; MASK_DIR=ROOT/'masks'/'corrected'
OUTPUT=ROOT/'reconstruction_v2'; FINAL_DIR=OUTPUT/'biharmonic_final';FAST_DIR=OUTPUT/'telea_preview';NS_DIR=OUTPUT/'navier_stokes';STAFF_DIR=OUTPUT/'biharmonic_staff_experimental';UNCERTAIN_DIR=OUTPUT/'uncertainty';REVIEW_DIR=OUTPUT/'review';AUDIT_DIR=OUTPUT/'audit'
for d in [FINAL_DIR,FAST_DIR,NS_DIR,STAFF_DIR,UNCERTAIN_DIR,REVIEW_DIR,AUDIT_DIR]:d.mkdir(parents=True,exist_ok=True)
USE_CORRECTED_MASKS=True
MODEL_PATH_OVERRIDE=None
CONFIDENCE=.20;TILE_SIZE=640;TILE_STRIDE=320;MODEL_IMAGE_SIZE=640
MASK_DILATION=1; CROP_MARGIN=48; FORCE_REBUILD_ALL=False
ENABLE_EXPERIMENTAL_STAFF=True

def find_image(folder,stem):
    hits=[p for p in image_files(folder) if p.stem.lower()==stem.lower()]
    return hits[0] if len(hits)==1 else None
print('Source:',SOURCE_DIR,'Mask:',MASK_DIR)


## Optional YOLO mask generation

Corrected masks are preferred. If a source lacks a corrected mask, set `USE_CORRECTED_MASKS=False`; the newest `best.pt` is then used to predict a mask.


In [ ]:
MODEL=None
def positions(n):
    if n<=TILE_SIZE:return [0]
    p=list(range(0,n-TILE_SIZE+1,TILE_STRIDE))
    if p[-1]!=n-TILE_SIZE:p.append(n-TILE_SIZE)
    return p

def load_model():
    global MODEL
    if MODEL is not None:return MODEL
    candidates=sorted((ROOT/'training_runs').rglob('best.pt'),key=lambda p:p.stat().st_mtime,reverse=True)
    path=Path(MODEL_PATH_OVERRIDE) if MODEL_PATH_OVERRIDE else (candidates[0] if candidates else None)
    if path is None or not path.exists():raise RuntimeError('No best.pt found.')
    if not torch.cuda.is_available():raise RuntimeError('Select a T4/L4 GPU runtime.')
    MODEL=YOLO(str(path));return MODEL

def predict_mask(gray):
    model=load_model();h,w=gray.shape;ph,pw=max(h,TILE_SIZE),max(w,TILE_SIZE)
    canvas=np.full((ph,pw,3),255,np.uint8);canvas[:h,:w]=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR);conf=np.zeros((ph,pw),np.float32)
    for y in positions(ph):
        for x in positions(pw):
            r=model.predict(canvas[y:y+TILE_SIZE,x:x+TILE_SIZE],conf=CONFIDENCE,imgsz=MODEL_IMAGE_SIZE,device=0,verbose=False,retina_masks=True)[0]
            if r.masks is None:continue
            for m,s in zip(r.masks.data.cpu().numpy(),r.boxes.conf.cpu().numpy()):
                m=cv2.resize(m,(TILE_SIZE,TILE_SIZE),interpolation=cv2.INTER_NEAREST)>.5;region=conf[y:y+TILE_SIZE,x:x+TILE_SIZE];region[m&(s>region)]=s
    return (conf[:h,:w]>=CONFIDENCE).astype(np.uint8)*255


## Reconstruction and uncertainty


In [ ]:
def prepare_mask(mask):
    mask=np.where(mask>0,255,0).astype(np.uint8)
    if MASK_DILATION>0:mask=cv2.dilate(mask,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*MASK_DILATION+1,2*MASK_DILATION+1)))
    return mask

def uncertainty_map(observed,mask):
    u=np.zeros_like(mask);dist=cv2.distanceTransform((mask>0).astype(np.uint8),cv2.DIST_L2,5);u[dist>6]=255
    # masked regions touching visible staff evidence are structurally sensitive
    near=cv2.dilate(staff_evidence(observed,mask).astype(np.uint8),np.ones((7,11),np.uint8))>0
    u[(mask>0)&near]=255
    return u

def review_panel(original,final,staff,mask,uncertainty):
    ov=cv2.cvtColor(original,cv2.COLOR_GRAY2BGR);layer=ov.copy();layer[mask>0]=(0,0,255);layer[uncertainty>0]=(0,165,255);ov=cv2.addWeighted(ov,.65,layer,.35,0)
    return np.hstack([cv2.cvtColor(original,cv2.COLOR_GRAY2BGR),cv2.cvtColor(final,cv2.COLOR_GRAY2BGR),cv2.cvtColor(staff,cv2.COLOR_GRAY2BGR),ov])


## Incremental processing


In [ ]:
records=[]
for source in image_files(SOURCE_DIR):
    gray=cv2.imread(str(source),0)
    if gray is None:continue
    mask_path=find_image(MASK_DIR,source.stem); mask=cv2.imread(str(mask_path),0) if mask_path else None
    if mask is None:
        if USE_CORRECTED_MASKS:continue
        mask=predict_mask(gray)
    if mask.shape!=gray.shape:print('Shape mismatch',source.name);continue
    mask=prepare_mask(mask)
    fp={'source_md5':md5(source),'mask_md5':md5(mask_path) if mask_path else None,'settings':object_hash({'dilation':MASK_DILATION,'margin':CROP_MARGIN,'staff':ENABLE_EXPERIMENTAL_STAFF})}
    audit_path=AUDIT_DIR/f'{source.stem}.json';old=json.loads(audit_path.read_text()) if audit_path.exists() else {}
    expected=[FINAL_DIR/f'{source.stem}.png',FAST_DIR/f'{source.stem}.png',NS_DIR/f'{source.stem}.png',UNCERTAIN_DIR/f'{source.stem}.png']
    if not FORCE_REBUILD_ALL and old.get('fingerprint')==fp and all(p.exists() for p in expected):print('Skipped',source.name);continue
    fast=crop_inpaint(gray,mask,'telea',CROP_MARGIN);ns=crop_inpaint(gray,mask,'navier_stokes',CROP_MARGIN);final=crop_inpaint(gray,mask,'biharmonic',CROP_MARGIN)
    staff=safe_staff_continuation(gray,mask,final) if ENABLE_EXPERIMENTAL_STAFF else final.copy();unc=uncertainty_map(gray,mask)
    for p,im in zip(expected,[final,fast,ns,unc]):cv2.imwrite(str(p),im)
    cv2.imwrite(str(STAFF_DIR/f'{source.stem}.png'),staff);cv2.imwrite(str(REVIEW_DIR/f'{source.stem}_review.png'),review_panel(gray,final,staff,mask,unc))
    audit={'created_utc':datetime.now(timezone.utc).isoformat(),'fingerprint':fp,'source':str(source),'mask':str(mask_path) if mask_path else 'YOLO','outside_mask_immutable':bool(np.array_equal(final[mask==0],gray[mask==0])),'uncertain_pixels':int(np.sum(unc>0))}
    audit_path.write_text(json.dumps(audit,indent=2));records.append(audit);print('Processed',source.name)
print('Processed pages:',len(records))


## Review recent outputs


In [ ]:
for p in sorted(REVIEW_DIR.glob('*_review.png'))[-10:]:
    im=cv2.imread(str(p));plt.figure(figsize=(24,9));plt.imshow(cv2.cvtColor(im,cv2.COLOR_BGR2RGB));plt.title(p.name+' | original | biharmonic | experimental staff | mask/uncertainty');plt.axis('off');plt.show()
